# Loading

In [1]:
import xml.etree.ElementTree as ET
import pandas as pd
import re

tree = ET.parse('detail.xml')
precincts = tree.getroot().find('VoterTurnout').find('Precincts')
print(tree.getroot().tag)
print(precincts) 

ElectionResult
<Element 'Precincts' at 0x000002C85B108BD0>


# Election Info

In [3]:
rows = []
for p in precincts.findall('Precinct'):
    rows.append({
        'name': p.get('name'),
        'registered': int(p.get('totalVoters')),
        'ballots_cast': int(p.get('ballotsCast'))
    })

df = pd.DataFrame(rows)
print(df.shape)
print(df.head())
print(df.tail()) 

(935, 3)
         name  registered  ballots_cast
0  Wayne  001         802           476
1  Wayne  002         775           438
2  Wayne  003         477           262
3  Wayne  004         786           391
4  Wayne  005         738           413
              name  registered  ballots_cast
930  FEDERAL - F03           0            30
931  FEDERAL - F04           0            14
932  FEDERAL - F06           0            64
933  FEDERAL - F08           0            20
934  FEDERAL - F11           0            39


In [4]:
df = df[~df['name'].str.contains('FEDERAL', case=False, na=False)].copy()
print(df.shape)  

(930, 3)


In [5]:
df['township'] = df['name'].str.extract(r'^(.*?)\s+\d+')
print(df['township'].nunique())  
print(sorted(df['township'].unique()))
print(df[df['township'].isna()]) 

9
['Addison', 'Bloomingdale', 'Downers Grove', 'Lisle', 'Milton', 'Naperville', 'Wayne', 'Winfield', 'York']
Empty DataFrame
Columns: [name, registered, ballots_cast, township]
Index: []


In [6]:
township_election = df.groupby('township')[['registered', 'ballots_cast']].sum().reset_index()
township_election['turnout'] = (township_election['ballots_cast'] / township_election['registered']) * 100

print(township_election.shape)  
print(township_election)

(9, 4)
        township  registered  ballots_cast    turnout
0        Addison       50373         23874  47.394437
1   Bloomingdale       69894         32750  46.856669
2  Downers Grove      106273         64204  60.414216
3          Lisle       81022         47628  58.784034
4         Milton       82186         50971  62.019079
5     Naperville       65008         34073  52.413549
6          Wayne       43982         22681  51.568824
7       Winfield       28170         15178  53.880014
8           York       89373         52208  58.415853


In [7]:
township_election.to_csv('dupage_township_election_2022_general.csv', index=False)
print('saved')

saved


In [9]:
print(df['registered'].sum())
print(township_election['registered'].sum())

616281
616281
